# WSJ 2027 - Registreringar över tid

Antal deltagare över tid, uppdelat på **rundresa** och **direktresa**.

**Datakälla:** Scoutnet API (projekt 43490)

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')

import requests
import pandas as pd
from datetime import date, datetime
from bokeh.plotting import figure, output_notebook, show, output_file, save
from bokeh.models import HoverTool, Label, NumeralTickFormatter
from bokeh.io import output_notebook
from scoutnet_secrets import SCOUTNET_API_ID, SCOUTNET_API_KEY

output_notebook()

# Fee IDs
DELTAGARE_RUNDRESA = '25694'
DELTAGARE_DIREKTRESA = '27561'
IST_RUNDRESA = '25696'
IST_EGEN_RESA = '25702'
CMT_FEES = {'25697', '25693'}
CAT_MAP = {
    DELTAGARE_RUNDRESA: 'Deltagare rundresa',
    DELTAGARE_DIREKTRESA: 'Deltagare direktresa',
    IST_RUNDRESA: 'IST rundresa',
    IST_EGEN_RESA: 'IST direktresa',
}

# Fetch participants
resp = requests.get(
    'https://www.scoutnet.se/api/project/get/participants',
    auth=(SCOUTNET_API_ID, SCOUTNET_API_KEY)
)
resp.raise_for_status()
data = resp.json()
print(f"Fetched {len(data['participants'])} participants")

Loading BokehJS ...

Fetched 1671 participants


In [2]:
# Build DataFrame with registration dates and travel type
rows = []
for mid, p in data['participants'].items():
    if p.get('cancelled'):
        continue
    fee_id = str(p.get('fee_id', ''))
    reg_date = p.get('registration_date', '')
    if not reg_date:
        continue
    try:
        reg_dt = datetime.strptime(reg_date, '%Y-%m-%d %H:%M:%S').date()
    except ValueError:
        continue
    if fee_id in CAT_MAP:
        category = CAT_MAP[fee_id]
    elif fee_id in CMT_FEES:
        category = 'CMT'
    else:
        continue
    rows.append({'date': reg_dt, 'category': category})

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['travel'] = df['category'].apply(
    lambda c: 'Rundresa' if 'rundresa' in c else ('Direktresa' if 'direktresa' in c else 'CMT'))

print(f"{len(df)} active registrations with dates")
print(f"\nPer kategori:")
print(df['category'].value_counts().to_string())

def cumulative_series(dataframe, filter_col, filter_val):
    """Build a cumulative daily series filled forward to today."""
    subset = dataframe[dataframe[filter_col] == filter_val].sort_values('date')
    daily = subset.groupby('date').size().cumsum().reset_index()
    daily.columns = ['date', 'count']
    date_range = pd.date_range(daily['date'].min(), pd.Timestamp.today())
    daily = daily.set_index('date').reindex(date_range).ffill().reset_index()
    daily.columns = ['date', 'count']
    daily['count'] = daily['count'].astype(int)
    return daily

1636 active registrations with dates

Per kategori:
category
Deltagare rundresa      1049
Deltagare direktresa     319
IST direktresa           171
IST rundresa              75
CMT                       22


In [3]:
# Cumulative registrations over time - Rundresa vs Direktresa (Bokeh)
p1 = figure(title='WSJ 2027 - Registreringar över tid (rundresa vs direktresa)',
            x_axis_type='datetime', width=900, height=450,
            tools='pan,wheel_zoom,box_zoom,reset,save')

colors = {'Rundresa': '#004B87', 'Direktresa': '#E8A838'}
for travel_type, color in colors.items():
    series = cumulative_series(df, 'travel', travel_type)
    p1.line(series['date'], series['count'], legend_label=travel_type,
            color=color, line_width=3)
    p1.scatter(series['date'], series['count'], color=color, size=0, alpha=0,
               legend_label=travel_type)  # invisible points for hover
    # End label
    p1.add_layout(Label(x=series['date'].iloc[-1], y=series['count'].iloc[-1],
                        text=f" {series['count'].iloc[-1]}", text_font_size='12pt',
                        text_font_style='bold', text_color=color))

# Total line
total_df = df[df['travel'] != 'CMT'].copy()
total_daily = total_df.sort_values('date').groupby('date').size().cumsum().reset_index()
total_daily.columns = ['date', 'count']
dr = pd.date_range(total_daily['date'].min(), pd.Timestamp.today())
total_daily = total_daily.set_index('date').reindex(dr).ffill().reset_index()
total_daily.columns = ['date', 'count']
total_daily['count'] = total_daily['count'].astype(int)
p1.line(total_daily['date'], total_daily['count'], legend_label='Totalt',
        color='#333333', line_width=1.5, line_dash='dashed', line_alpha=0.7)
p1.add_layout(Label(x=total_daily['date'].iloc[-1], y=total_daily['count'].iloc[-1],
                     text=f" {total_daily['count'].iloc[-1]}", text_font_size='11pt',
                     text_color='#333333'))

p1.add_tools(HoverTool(tooltips=[('Datum', '@x{%F}'), ('Antal', '@y')],
                        formatters={'@x': 'datetime'}, mode='vline'))
p1.yaxis.axis_label = 'Antal registrerade (kumulativt)'
p1.xaxis.axis_label = 'Datum'
p1.legend.location = 'top_left'
p1.legend.click_policy = 'hide'
p1.grid.grid_line_alpha = 0.3

show(p1)

# Also save standalone HTML
output_file('/config/notebooks/wsj27/wsj_registrations_over_time.html')
save(p1)
print("Saved: wsj_registrations_over_time.html")

Saved: wsj_registrations_over_time.html


In [4]:
# Detailed breakdown: 4 categories (Bokeh)
p2 = figure(title='WSJ 2027 - Registreringar per kategori',
            x_axis_type='datetime', width=900, height=450,
            tools='pan,wheel_zoom,box_zoom,reset,save')

detail_colors = {
    'Deltagare rundresa': '#004B87',
    'Deltagare direktresa': '#E8A838',
    'IST rundresa': '#66A3CC',
    'IST direktresa': '#F0C87A',
}

for cat, color in detail_colors.items():
    subset = df[df['category'] == cat]
    if len(subset) == 0:
        continue
    series = cumulative_series(df, 'category', cat)
    p2.line(series['date'], series['count'], legend_label=cat,
            color=color, line_width=2.5)
    p2.add_layout(Label(x=series['date'].iloc[-1], y=series['count'].iloc[-1],
                        text=f" {series['count'].iloc[-1]}", text_font_size='11pt',
                        text_font_style='bold', text_color=color))

p2.add_tools(HoverTool(tooltips=[('Datum', '@x{%F}'), ('Antal', '@y')],
                        formatters={'@x': 'datetime'}, mode='vline'))
p2.yaxis.axis_label = 'Antal registrerade (kumulativt)'
p2.xaxis.axis_label = 'Datum'
p2.legend.location = 'top_left'
p2.legend.click_policy = 'hide'
p2.grid.grid_line_alpha = 0.3

show(p2)

# Also save standalone HTML
output_file('/config/notebooks/wsj27/wsj_registrations_detail.html')
save(p2)
print("Saved: wsj_registrations_detail.html")

Saved: wsj_registrations_detail.html
